# Escaping the Morlocks · Lab 01: Delta Lake — follow the transaction log

> Three commits. Three clocks. One log that can reconstruct the file set.

## Mission

Replay the incident, read versions 0/1/2 with `versionAsOf`, and prove that an old business timestamp does not make a row appear in the old snapshot.

## Same incident, same expected states

| State | Mutation | Rows |
|---|---|---:|
| Baseline | Ada + Babbage | 2 |
| Late arrival | Update Babbage; append Larry with an older `event_time` | 3 |
| Correction | Delete Ada | 2 |

Each run creates a random temporary table. No Docker, named production table, remote catalog or Databricks workspace is required for these local OSS labs. Install `requirements.txt`; Spark downloads its pinned connector packages from Maven Central on first run.

## 1 · Start Delta 3.3.2 on Spark 3.5.9

Path-based Delta keeps this lab independent of any catalog.

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
b=(SparkSession.builder.master("local[2]").appName("Morlocks-Delta").config("spark.ui.enabled","false").config("spark.sql.shuffle.partitions","2").config("spark.default.parallelism","2").config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension").config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog"))
spark=configure_spark_with_delta_pip(b).getOrCreate(); spark.sparkContext.setLogLevel("ERROR")
print("Spark",spark.version,"Delta 3.3.2")

## 2 · Write the shared three-state history

We keep every commit and never run `VACUUM`: cleanup would undermine the very history we are testing.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, shutil, tempfile
from pyspark.sql import functions as F, types as T
ROOT=Path(tempfile.mkdtemp(prefix="morlocks-LAB-" )).resolve()
PATH=(ROOT/"events").as_uri()
SCHEMA=T.StructType([T.StructField("id",T.LongType(),False),T.StructField("name",T.StringType(),False),T.StructField("segment",T.StringType(),False),T.StructField("balance",T.DoubleType(),False),T.StructField("event_time",T.TimestampType(),False)])
BASE=[(1,"Ada","bronze",100.,datetime(2026,10,24,8,0)),(2,"Babbage","silver",200.,datetime(2026,10,24,8,5))]
LATE=[(2,"Babbage Lovelace","gold",250.,datetime(2026,10,24,8,5)),(99,"Larry Latecomer","time-traveller",42.42,datetime(2026,10,20,9,15))]
def df(rows): return spark.createDataFrame(rows,SCHEMA)
def proof(frame):
 rows=frame.select("id","name","segment","balance",F.date_format("event_time","yyyy-MM-dd HH:mm:ss").alias("event_time")).orderBy("id").collect()
 raw="\n".join("|".join(str(v) for v in r) for r in rows)
 return len(rows),hashlib.sha256(raw.encode()).hexdigest(),rows
from delta.tables import DeltaTable
df(BASE).write.format("delta").mode("overwrite").save(PATH)
MERGE=f"""MERGE INTO delta.`{PATH}` t USING incoming s ON t.id=s.id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *"""
# A SQL view lets MERGE retain one atomic Delta commit.
df(LATE).createOrReplaceTempView("incoming")
spark.sql(MERGE)
spark.sql(f"DELETE FROM delta.`{PATH}` WHERE id=1")
h=DeltaTable.forPath(spark,PATH).history().select("version","timestamp","operation").orderBy("version")
h.show(truncate=False)

## 3 · Travel to each native version

In [ ]:
def at(v): return spark.read.format("delta").option("versionAsOf",v).load(PATH)
states=[at(0),at(1),at(2)]
for label,state in zip(("baseline","late arrival","correction"),states): print(label,proof(state)[:2])
assert [x.count() for x in states]==[2,3,2]
assert states[0].where("id=99").count()==0 and states[1].where("id=99").count()==1
assert states[2].where("id=1").count()==0
commit=h.where(F.col("version")==1).select("timestamp").first()[0]
larry=states[1].where("id=99").select("event_time").first()[0]
print("Larry event_time",larry,"first visible in version 1; commit",commit)
assert larry < commit
print("PASS: event time is data; Delta version is table history.")

## Conclusions · Delta

**Strength:** the transaction log reconstructs the active file set at a table version. Version travel is precise and table-local.

**Pitfalls:** versions are not portable global IDs; timestamp travel derives from log-file timestamps and can change after copying a directory. `VACUUM` can remove data needed by old versions. A log is not a backup.

**Best practice:** persist table identity + exact version for reproducibility; align log and data retention to audit/rollback promises. This lab never vacuums.

In [ ]:
spark.stop(); shutil.rmtree(ROOT,ignore_errors=True); print("PASS: temporary lab cleaned up")